# 05_tune_nsga2: Step 5 - tune NSGA-II hyperparameters with PSO, GA and HHO (Lyu et al. used Optuna).

```
Search space (paper Table A.8 widened, + setpoint mutation sigma):
  pop_size 10-100, mutation prob 0.05-1, wall mutation sd eta_m 0.2-6 m,
  crossover prob 0.1-1, setpoint mutation sd sigma_t 0.05-1.5 degC.
Fitness = -mean normalised hypervolume over 3 tuning seeds (101-103) at a FIXED
budget of function evaluations, so a bigger population cannot win by simply
spending more evaluations. Tuning seeds are disjoint from the final seeds 1-30.
Script asli: `scripts/05_tune_nsga2.py` [--budget 50] [--mh-seeds 2] [--nsga-evals 3000] — parameter kini diatur di sel `args` di bawah.
```

Dikonversi dari `scripts/05_tune_nsga2.py`. Jalankan dari folder `notebooks/`.

In [1]:
from types import SimpleNamespace
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.layout import case_study_1, case_study_2
from obep_mlpaper.metaheuristics import ALGORITHMS, Param, SearchSpace
from obep_mlpaper.nsga import run_nsga2
from obep_mlpaper.surrogate import Surrogate

# Parameters (were command-line arguments in the script; edit here)
args = SimpleNamespace(budget=50, mh_seeds=2, nsga_evals=3000, pop=10)

SPACE = SearchSpace([
    Param("pop_size", 10, 100, "int"),
    Param("p_m", 0.05, 1.0),
    Param("eta_m", 0.2, 6.0, "log"),
    Param("p_c", 0.1, 1.0),
    Param("sigma_t", 0.05, 1.5, "log"),
])
TUNE_SEEDS = (101, 102, 103)
sur = Surrogate.load(ROOT / "outputs" / "models" / "surrogate.joblib").single_thread()
out = ROOT / "outputs" / "nsga2"
out.mkdir(parents=True, exist_ok=True)

In [2]:
result = {}
runs, traces = [], {}
for sc in (case_study_1(), case_study_2()):
    def fitness(hp, sc=sc):
        hvs = Parallel(n_jobs=2)(delayed(run_nsga2)(sc, sur, hp, s, args.nsga_evals) for s in TUNE_SEEDS)
        return -float(np.mean([r["hv"] for r in hvs]))

    for algo, fn in ALGORITHMS.items():
        for seed in range(args.mh_seeds):
            t0 = time.time()
            kw = {"PSO": "n_particles", "GA": "pop_size", "HHO": "n_hawks"}[algo]
            r = fn(fitness, SPACE, budget=args.budget, seed=seed, **{kw: args.pop})
            runs.append(dict(case=sc.name, algo=algo, seed=seed, best_hv=-r["best_f"],
                             minutes=(time.time() - t0) / 60, params=json.dumps(r["best_params"])))
            traces[f"{sc.name}_{algo}_s{seed}"] = -r["trace"]
            print(f"{sc.name} {algo} seed={seed}: HV={-r['best_f']:.4f} {r['best_params']} "
                  f"({(time.time() - t0) / 60:.1f} min)", flush=True)

runs = pd.DataFrame(runs)
runs.to_csv(out / "nsga2_tuning_runs.csv", index=False)
L = max(len(t) for t in traces.values())
pd.DataFrame({k: np.pad(t, (0, L - len(t)), mode="edge") for k, t in traces.items()}).to_csv(
    out / "nsga2_tuning_convergence.csv", index_label="evaluation")
for case, grp in runs.groupby("case"):
    result[case] = {a: json.loads(grp[grp.algo == a].sort_values("best_hv").iloc[-1].params) for a in ALGORITHMS}
(out / "nsga2_tuned_hyperparameters.json").write_text(json.dumps(result, indent=2))
print(runs.groupby(["case", "algo"]).best_hv.agg(["mean", "std", "max"]).round(4))

case1_simple PSO seed=0: HV=1.0421 {'pop_size': 42, 'p_m': 0.24211115968045, 'eta_m': 0.5281117935596886, 'p_c': 0.8003497173048788, 'sigma_t': 0.3682078277971319} (1.0 min)


case1_simple PSO seed=1: HV=1.0464 {'pop_size': 64, 'p_m': 0.05876321956441891, 'eta_m': 1.9002981450835421, 'p_c': 0.7215284676580384, 'sigma_t': 0.8080935071805523} (0.8 min)


case1_simple GA seed=0: HV=1.0392 {'pop_size': 59, 'p_m': 0.4105294740016725, 'eta_m': 0.31368465952887165, 'p_c': 0.7239435484232933, 'sigma_t': 0.28479622372374286} (0.9 min)


case1_simple GA seed=1: HV=1.0439 {'pop_size': 57, 'p_m': 0.19757048780948872, 'eta_m': 1.4284749856886374, 'p_c': 0.8184102574985498, 'sigma_t': 0.38827825750229855} (0.8 min)


case1_simple HHO seed=0: HV=1.0431 {'pop_size': 30, 'p_m': 0.17003819390632038, 'eta_m': 0.2138322526038332, 'p_c': 0.12206927256820709, 'sigma_t': 0.241551708876658} (1.2 min)


...

case2_watson GA seed=0: HV=1.2356 {'pop_size': 11, 'p_m': 0.1701348199095036, 'eta_m': 2.675459686704012, 'p_c': 0.6554370472729145, 'sigma_t': 0.33833014165092246} (1.3 min)


case2_watson GA seed=1: HV=1.2084 {'pop_size': 59, 'p_m': 0.12916629376218486, 'eta_m': 1.6551981625794754, 'p_c': 0.7945728289572451, 'sigma_t': 0.6764702609055137} (0.8 min)


case2_watson HHO seed=0: HV=1.2541 {'pop_size': 12, 'p_m': 0.059363410296500614, 'eta_m': 3.7035598004546038, 'p_c': 0.5511718609838577, 'sigma_t': 0.6291532599967401} (1.7 min)


case2_watson HHO seed=1: HV=1.2229 {'pop_size': 40, 'p_m': 0.0710545909821538, 'eta_m': 2.2073386436510214, 'p_c': 0.8720295973335862, 'sigma_t': 0.955065180955524} (1.3 min)


                     mean     std     max
case         algo                        
case1_simple GA    1.0416  0.0033  1.0439
             HHO   1.0416  0.0021  1.0431
             PSO   1.0443  0.0030  1.0464
case2_watson GA    1.2220  0.0192  1.2356
             HHO   1.2385  0.0221  1.2541
             PSO   1.2379  0.0002  1.2380
